# Démo 4 : profiler, mixed precision et débogage

Objectif : mesurer où passe le temps avec le profiler, comparer une époque avec et sans `autocast`, localiser un NaN avec la détection d'anomalies, et surveiller les activations avec un hook.

**Données** : Fashion-MNIST (Zalando Research, licence MIT), fourni par torchvision : 70 000 images de vêtements en niveaux de gris, 28 x 28 pixels, 10 classes.

**Ce que vous allez voir**

- mesurer où passe le temps avec le profiler ;
- comparer une époque en float32 et en mixed precision ;
- déboguer : localiser un NaN, surveiller les activations, repérer des dying ReLU ; puis la reproductibilité et DataParallel.

**Durée prévue** : 10 min.

*Les sorties enregistrées dans ce notebook viennent d'une exécution dans Google Colab sur GPU A100, avec la graine fixée. Sur un autre GPU ou sur CPU, certaines valeurs peuvent légèrement différer : PyTorch ne garantit pas des résultats identiques d'une plateforme à l'autre.*

## 0. Installation

Cette cellule écrit `requirements-colab.txt` (les versions de la formation, alignées sur le runtime Colab), puis l'installe avec uv. Les paquets déjà présents à la bonne version ne sont pas réinstallés. Hors de Colab, elle n'installe rien : l'environnement uv du projet contient déjà ces versions.

In [ ]:
import importlib.util
from pathlib import Path

Path("requirements-colab.txt").write_text("""torch==2.11.0
torchvision==0.26.0
numpy==2.1.3
matplotlib==3.10.0
pillow==11.3.0
scikit-learn==1.6.1
pandas==2.2.2; python_version < "3.13"
pandas==2.2.3; python_version >= "3.13"
pyarrow>=14
tensorboard==2.21.0
torchviz==0.0.3
onnx==1.23.0
onnxruntime==1.30.0
onnxscript==0.7.2
torchserve==0.12.0
torch-model-archiver==0.12.0
""")

if importlib.util.find_spec("google.colab") is not None:
    %pip install -q --progress-bar off uv
    !uv pip install --system -q -r requirements-colab.txt
else:
    print("Hors Colab : environnement uv du projet, rien à installer.")

Dans Colab, seuls les paquets absents du runtime ou à une autre version sont téléchargés. Si torch ou torchvision venaient à être réinstallés, redémarrez la session (**Exécution > Redémarrer la session**) puis relancez le notebook.

## 1. Télécharger Fashion-MNIST

**On télécharge Fashion-MNIST une fois pour toutes dans le dossier `data`.**

Le code :
- `datasets.FashionMNIST("data", train=..., download=True)` : télécharge les deux jeux s'ils ne sont pas déjà
  là ;
- `%%capture` : masque la barre de progression.

In [ ]:
%%capture
from torchvision import datasets

datasets.FashionMNIST("data", train=True, download=True)
datasets.FashionMNIST("data", train=False, download=True)

Les fichiers sont dans le dossier `data` : les cellules suivantes les retrouvent sans rien retélécharger.

## 2. Les images en mémoire sur le device

**Toutes les images sont chargées d'un coup sur le GPU.**

Fashion-MNIST tient en mémoire. En plaçant tous les tenseurs sur le GPU une fois pour toutes,
l'entraînement ne relit plus aucun fichier et ne copie plus rien : il ne reste que le calcul. C'est ce
qu'on veut ici, puisqu'on va mesurer des temps de calcul. Dans un projet, quand les données ne tiennent pas en
mémoire, on garde le Dataset et le DataLoader de ce matin.

Le code :
- `train.data` : les 60 000 images, en entiers de 0 à 255 ; `.float().div(255)` les ramène entre 0 et 1 ;
- `.unsqueeze(1)` : ajoute la dimension du canal, pour obtenir (images, 1, 28, 28) ;
- `.to(device)` : tout va sur le GPU, images et labels.

In [ ]:
import torch
from torchvision import datasets

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
train = datasets.FashionMNIST("data", train=True)
test = datasets.FashionMNIST("data", train=False)
x_train = train.data.float().div(255).unsqueeze(1).to(device)
y_train = train.targets.to(device)
x_test = test.data.float().div(255).unsqueeze(1).to(device)
y_test = test.targets.to(device)
print(x_train.shape, x_test.shape, x_train.device)

Les 60000 images d'entraînement et les 10000 images de test sont sur le GPU (`cuda:0`) : les mesures porteront sur le calcul seul.

## 3. Un CNN

**Une classe qui construit un CNN de la largeur voulue.**

La même architecture servira à plusieurs mesures ; seule sa largeur (le nombre de canaux) change.

Le code :
- `bloc1` et `bloc2` : deux blocs convolution, ReLU, pooling, avec `largeur` canaux, puis deux fois plus ;
- `classifieur` : `nn.Flatten()`, puis deux couches linéaires jusqu'aux 10 scores ;
- `forward` : les données traversent `bloc1`, `bloc2`, puis `classifieur`.

In [ ]:
import torch.nn as nn


class CNN(nn.Module):
    def __init__(self, largeur=32):
        super().__init__()
        self.bloc1 = nn.Sequential(nn.Conv2d(1, largeur, 3, padding=1), nn.ReLU(), nn.MaxPool2d(2))
        self.bloc2 = nn.Sequential(nn.Conv2d(largeur, 2 * largeur, 3, padding=1), nn.ReLU(), nn.MaxPool2d(2))
        self.classifieur = nn.Sequential(
            nn.Flatten(), nn.Linear(2 * largeur * 7 * 7, 128), nn.ReLU(), nn.Linear(128, 10)
        )

    def forward(self, x):
        return self.classifieur(self.bloc2(self.bloc1(x)))

Rien à afficher : la classe est définie. Le paramètre `largeur` fixe le nombre de canaux de la première convolution ; la seconde en a deux fois plus.

## 4. Le profiler

**Le profiler mesure le temps passé dans chaque opération.**

Avant d'optimiser, on mesure. Le profiler dit quelles opérations coûtent, sur le CPU et sur le GPU,
pendant un forward et un backward.

Le code :
- une première passe, hors mesure : le GPU charge ses programmes au premier appel, ce qui fausserait la
  mesure ;
- `with profile(activities=...) as prof:` : tout ce qui s'exécute dans le bloc est mesuré ;
- `prof.key_averages().table(sort_by=tri, row_limit=6)` : les 6 opérations les plus coûteuses, triées par
  temps GPU.

In [ ]:
from torch.profiler import ProfilerActivity, profile

torch.manual_seed(0)
model = CNN(256).to(device)
nn.CrossEntropyLoss()(model(x_train[:256]), y_train[:256]).backward()
activites = [ProfilerActivity.CPU] + ([ProfilerActivity.CUDA] if device.type == "cuda" else [])
tri = "self_cuda_time_total" if device.type == "cuda" else "self_cpu_time_total"
with profile(activities=activites) as prof:
    nn.CrossEntropyLoss()(model(x_train[:256]), y_train[:256]).backward()
print(prof.key_averages().table(sort_by=tri, row_limit=6))

Sur le GPU, le backward de la convolution prend 32.84 % du temps, celui du max pooling 17.97 %, le forward de la convolution 15.67 %. Le backward coûte plus que le forward. Les noyaux dont le nom contient `tf32` montrent que l'A100 calcule déjà les convolutions float32 en TF32, un format réduit. Total : 9.691 ms de GPU pour un forward et un backward sur 256 images.

**À retenir** : mesurer avant d'optimiser ; le profiler dit où agir.

## 5. Une époque, avec ou sans autocast (fourni)

**Une fonction qui entraîne une époque et renvoie sa durée, en float32 ou en mixed precision.**

La **mixed precision** fait les gros calculs (convolutions, couches linéaires) en float16, deux fois
plus petit que le float32 : moins de mémoire, et des calculs plus rapides sur les GPU récents. Les calculs
sensibles restent en float32.

Le code (à survoler) :
- `torch.autocast(device_type="cuda", dtype=torch.float16, enabled=amp)` : le forward et la loss passent en
  mixed precision quand `amp` est vrai ;
- `torch.amp.GradScaler` : multiplie la loss avant le backward, pour que les petits gradients ne tombent pas à
  zéro en float16 ; `scaler.step()` et `scaler.update()` remplacent `optimizer.step()` ;
- `torch.cuda.synchronize()` : attend la fin des calculs du GPU avant de lire l'heure.

In [ ]:
import time


def epoque(model, amp):
    optimizer = torch.optim.Adam(model.parameters(), lr=1e-3)
    scaler = torch.amp.GradScaler("cuda", enabled=amp and device.type == "cuda")
    if device.type == "cuda":
        torch.cuda.synchronize()
    debut = time.perf_counter()
    for indices in torch.randperm(len(x_train), device=device).split(256):
        with torch.autocast(device_type="cuda", dtype=torch.float16, enabled=amp and device.type == "cuda"):
            loss = nn.CrossEntropyLoss()(model(x_train[indices]), y_train[indices])
        optimizer.zero_grad()
        scaler.scale(loss).backward()
        scaler.step(optimizer)
        scaler.update()
    if device.type == "cuda":
        torch.cuda.synchronize()
    return time.perf_counter() - debut

Rien à afficher : la fonction est définie. `torch.cuda.synchronize()` attend la fin des calculs du GPU avant de lire l'heure : sans cela, on mesurerait seulement le temps de lancement des opérations.

## 6. Comparer les durées

**Une époque en float32, une en mixed precision, sur un CNN large.**

On chauffe d'abord le GPU dans les deux modes, hors mesure, pour ne comparer que le calcul.

Le code :
- `CNN(256)` : 256 puis 512 canaux ; sur un petit modèle, le temps passe à lancer les opérations, pas à
  calculer, et la mixed precision n'aurait rien à accélérer ;
- deux appels à `epoque`, avec `amp=False` puis `amp=True`.

In [ ]:
torch.manual_seed(0)
for amp in (False, True):
    epoque(CNN(256).to(device), amp)
print(f"float32          : {epoque(CNN(256).to(device), amp=False):.2f} s")
print(f"mixed precision  : {epoque(CNN(256).to(device), amp=True):.2f} s")

1.96 s en float32, 1.50 s en mixed precision : près d'un quart de temps en moins, pour le même modèle et les mêmes données. Le gain reste modeste, car les convolutions float32 profitaient déjà du TF32 sur l'A100.

**Essayez** : refaites la mesure avec `CNN(32)` ; sur un petit modèle, le gain disparaît.

## 7. Localiser un NaN

**Un NaN (« not a number ») peut apparaître, puis contaminer tout l'entraînement.**

Une division 0 / 0, un logarithme de 0... produit un NaN, qui se propage à tous les calculs
suivants, puis aux poids. La **détection d'anomalies** arrête le backward sur la première opération qui
renvoie un NaN, et la nomme.

Le code :
- `(x / x)` avec `x = [0, 1]` : 0 / 0 donne un NaN ;
- `torch.autograd.detect_anomaly()` : active la détection pour le bloc ; elle ralentit les calculs, on ne
  l'active que pour déboguer ;
- `warnings.catch_warnings(action="ignore")` : masque un avertissement très long, pour ne garder que l'erreur.

In [ ]:
import warnings

x = torch.tensor([0.0, 1.0], requires_grad=True)
# on masque l'avertissement, qui répète la pile d'appels complète
with warnings.catch_warnings(action="ignore"), torch.autograd.detect_anomaly():
    try:
        (x / x).sum().backward()
    except RuntimeError as err:
        print(err)

L'erreur nomme l'opération : `DivBackward0`, le backward de la division, qui renvoie un NaN pour sa deuxième entrée (`1th output`, en comptant à partir de 0), le dénominateur. En 0, la dérivée de x / x par rapport au dénominateur vaut -x / x², soit 0 / 0.

**À retenir** : la détection d'anomalies ralentit tout ; on l'active le temps de trouver le bug.

## 8. Surveiller les activations avec un hook

**Un hook est une fonction que PyTorch appelle automatiquement après le forward d'une couche.**

On peut observer l'intérieur du réseau sans modifier son code : l'ordre de grandeur des sorties,
la part de zéros, la présence de NaN.

Le code :
- `stats(module, entrees, sortie)` : la fonction appelée, qui affiche l'écart-type, le maximum et la part de
  valeurs nulles de la sortie ;
- `couche.register_forward_hook(stats)` : on l'accroche à chaque couche des trois blocs ;
- `model(x_train[:64])` : un forward sur 64 images déclenche les affichages.

In [ ]:
def stats(module, entrees, sortie):
    nom = module.__class__.__name__
    print(f"{nom:10s} std {sortie.std().item():6.3f}  max {sortie.abs().max().item():6.3f}  "
          f"nulles {(sortie == 0).float().mean().item():.0%}")


model = CNN(32).to(device)
for couche in list(model.bloc1) + list(model.bloc2) + list(model.classifieur):
    couche.register_forward_hook(stats)
with torch.no_grad():
    model(x_train[:64])

Après chaque ReLU, environ la moitié des valeurs sont nulles (43 %, 51 %, 54 %) : c'est le fonctionnement normal. L'écart-type diminue de couche en couche, de 0.346 à 0.047, sans explosion.

**À retenir** : un hook observe le réseau sans modifier son code ; on le retire avec la méthode `remove()` de l'objet renvoyé.

## 9. Provoquer des dying ReLU

**Une ReLU « morte » renvoie toujours 0 : son gradient est nul, elle n'apprend plus.**

Si toutes les entrées d'une ReLU sont négatives, elle renvoie 0 partout, et plus aucune correction
ne lui parvient. On le provoque exprès pour voir ce que le hook affiche alors.

Le code :
- `model.bloc1[0].bias.fill_(-5.0)` : un biais très négatif sur la première convolution ;
- le même forward : les hooks sont toujours accrochés.

Prédisez la part de zéros à la sortie de la première ReLU.

In [ ]:
with torch.no_grad():
    model.bloc1[0].bias.fill_(-5.0)
    model(x_train[:64])

La première ReLU renvoie 100 % de zéros : ses neurones sont morts. Le maximum de 6.096 affiché pour la convolution est une valeur absolue : ses sorties sont négatives. La deuxième convolution ne voit plus l'image, et comme le gradient de la ReLU est nul sous zéro, ces neurones ne peuvent plus apprendre.

**Piège fréquent** : un learning rate trop grand peut pousser des neurones dans cette zone morte ; surveiller la part de zéros aide à le repérer.

## 10. Reproductibilité

**Même seed, même résultat.**

Pour comparer deux réglages, il faut que tout le reste soit identique, y compris le tirage des
poids initiaux. Fixer la **seed** du générateur aléatoire rend les tirages reproductibles.

Le code :
- `torch.manual_seed(0)` avant chaque création de modèle, puis comparaison des poids de la première couche ;
- `torch.backends.cudnn.benchmark = False` et `torch.use_deterministic_algorithms(True, warn_only=True)` :
  demandent en plus des algorithmes déterministes sur GPU, au prix d'un peu de vitesse.

In [ ]:
torch.manual_seed(0)
poids_1 = CNN(32).bloc1[0].weight
torch.manual_seed(0)
poids_2 = CNN(32).bloc1[0].weight
print("mêmes poids :", torch.equal(poids_1, poids_2))
torch.backends.cudnn.benchmark = False
torch.use_deterministic_algorithms(True, warn_only=True)

`mêmes poids : True` : avec la même seed, deux initialisations donnent exactement les mêmes poids. Les deux dernières lignes demandent en plus à cuDNN des algorithmes déterministes, pour que l'entraînement lui-même soit reproductible sur GPU.

**À retenir** : fixer la seed avant chaque expérience que l'on veut comparer.

## 11. DataParallel

**Avec plusieurs GPU, `nn.DataParallel` répartit chaque lot entre les cartes.**

Chaque GPU traite une partie du lot avec une copie du modèle, puis les sorties sont réunies. La
boucle d'entraînement ne change pas.

Le code :
- `torch.cuda.device_count()` : le nombre de GPU ;
- `nn.DataParallel(model)` : enveloppe le modèle, seulement s'il y a plus d'un GPU.

In [ ]:
print("GPU disponibles :", torch.cuda.device_count())
model = CNN(32)
if torch.cuda.device_count() > 1:
    model = nn.DataParallel(model)
print(type(model).__name__)

Un seul GPU dans ce runtime : le modèle reste un `CNN`. Sur une machine à plusieurs GPU, il serait enveloppé dans un `DataParallel`, sans rien changer à la boucle d'entraînement.

## À retenir de cette démo

- Le profiler montre les opérations coûteuses ; la mixed precision accélère les calculs et allège la mémoire, surtout sur de gros modèles.
- La détection d'anomalies nomme l'opération qui produit un NaN ; un forward hook surveille les sorties de chaque couche.
- Une seed fixe rend les résultats reproductibles ; DataParallel répartit un lot entre plusieurs GPU.